In [43]:
# Importing Functions from Transformer_Architecture
%run Transformer_Architecture.ipynb

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]])
tensor([[ 3, 18,  8, 11, 10, 10, 10,  3, 15,  0],
        [24, 23,  0,  0,  0,  0,  0,  0,  0,  0],
        [11,  1,  7,  0,  0,  0,  0,  0,  0,  0],
        [12,  4,  8, 13,  0,  0,  0,  0,  0,  0]])
tensor([[[ 0.4829, -0.9607, -3.2539,  ..., -1.4467, -0.7034,  2.4647],
         [ 1.6725, -0.0481, -2.3087,  ...,  0.6365,  0.9097, -1.5111],
         [ 1.2013, -1.6087, -0.8679,  ..., -0.3661,  2.0350, -0.3882],
         ...,
         [ 1.9824, -1.6456, -2.5990,  ..., -0.0000, -2.1989,  0.5339],
         [-1.0326,  0.1507,  0.6481,  ..., -1.3696, -2.4282,  1.4916],
         [-3.3713,  1.2514, -2.5802,  ...,  1.4124, -1.8299,  2.3367]],

        [[-0.0000, -1.3213, -1.2499,  ..., -0.1477, -0.5285,  0.6674],
         [ 2.0862, -1.6508, -1.7526,  ...,  3.9291,  1.8358, -1.5921],
         [-0.8872, -1.1269, -1.3413,  ..., -0.6551,  1

In [44]:
# Multi Token Prediction
# input_ids = [10, 20, 30, 40, 50]
# Assume the final hidden states are: h0, h1, h2, h3, h4

# Head 0: normal next-token head
# h0 → predict 20
# h1 → predict 30
# h2 → predict 40
# h3 → predict 50

# Head 1: predict two tokens ahead
# h0 → predict 30
# h1 → predict 40
# h2 → predict 50

# Head 2: predict three tokens ahead
# h0 → predict 40
# h1 → predict 50

# Head 3: predict four tokens ahead
# h0 → predict 50

# Note: Multi Token Prediction is only during training to improve training performance where as during inference its 
# normal inference


In [45]:
# Multi Token Prediction

@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 6
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # Multi-token prediction
    num_future_tokens: int = 4
    mtp_loss_weight: float = 1.0

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [46]:
class MTPGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = TokenEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            DecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        # Original MTP paper style:
        # independent output heads on top of the shared trunk.
        #
        # head 0 predicts t+1
        # head 1 predicts t+2
        # head 2 predicts t+3
        # ...
        self.mtp_heads = nn.ModuleList([
            nn.Linear(config.d_model, config.vocab_size, bias=False)
            for _ in range(config.num_future_tokens)
        ])

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:      [B, T]
        attention_mask: [B, T]
        labels:         [B, T]
    
        returns:
            logits: first-head next-token logits, [B, T, vocab_size]
            mtp_logits: list of logits from all heads
            loss: main next-token loss + auxiliary MTP loss
        """
    
        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.config.max_seq_len
    
        x = self.embedding(input_ids)
        # [B, T, D]
    
        for block in self.blocks:
            x = block(
                x,
                attention_mask=attention_mask,
            )
    
        x = self.final_ln(x)
        # [B, T, D]
    
        mtp_logits = [
            head(x)
            for head in self.mtp_heads
        ]
        # each: [B, T, vocab_size]
    
        # Head 0 is the normal next-token prediction head.
        logits = mtp_logits[0]
        # [B, T, vocab_size]
    
        loss = None
        main_loss = None
        aux_loss = None
        losses_by_depth = {}
    
        if labels is not None:
            # -------------------------
            # Main next-token loss
            # head 0: position t predicts token t+1
            # -------------------------
            main_shift_logits = logits[:, :-1, :]
            main_shift_labels = labels[:, 1:]
    
            main_loss = F.cross_entropy(
                main_shift_logits.reshape(-1, self.config.vocab_size), 
                main_shift_labels.reshape(-1),
                ignore_index=-100,
            )
    
            losses_by_depth["main_next_token_loss_depth_1"] = main_loss.detach()
    
            # -------------------------
            # Auxiliary MTP losses
            # head 1 predicts t+2
            # head 2 predicts t+3
            # ...
            # -------------------------
            aux_losses = []
    
            for head_idx in range(1, len(mtp_logits)):
                depth = head_idx + 1
                # head_idx=1 → depth=2 → predict token t+2
                # head_idx=2 → depth=3 → predict token t+3
    
                depth_logits = mtp_logits[head_idx]
    
                if seq_len <= depth:
                    continue
    
                shift_logits = depth_logits[:, :-depth, :] # Excellent very simple
                shift_labels = labels[:, depth:]
    
                valid = shift_labels != -100
    
                if valid.any():
                    loss_depth = F.cross_entropy(
                        shift_logits.reshape(-1, self.config.vocab_size),
                        shift_labels.reshape(-1),
                        ignore_index=-100,
                    )
    
                    aux_losses.append(loss_depth)
                    losses_by_depth[f"mtp_aux_loss_depth_{depth}"] = loss_depth.detach()
    
            if len(aux_losses) > 0:
                aux_loss = torch.stack(aux_losses).mean()
                loss = main_loss + self.config.mtp_loss_weight * aux_loss
            else:
                aux_loss = torch.tensor(
                    0.0,
                    device=input_ids.device,
                    dtype=main_loss.dtype,
                )
                loss = main_loss
    
        return {
            "logits": logits,
            "mtp_logits": mtp_logits,
            "loss": loss,
            "main_loss": main_loss,
            "aux_loss": aux_loss,
            "losses_by_depth": losses_by_depth,
        }

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
        eos_token_id: int | None = None,
    ):
        """
        Standard autoregressive generation.

        We use mtp_heads[0], the normal next-token head.
        The additional MTP heads are training-time auxiliary heads in this simple version. Very Important
        """

        self.eval()

        for _ in range(max_new_tokens):
            input_ids_cond = input_ids[:, -self.config.max_seq_len:]

            attention_mask = (
                input_ids_cond != self.config.pad_token_id
            ).long()

            outputs = self(
                input_ids=input_ids_cond,
                attention_mask=attention_mask,
                labels=None,
            )

            logits = outputs["logits"]
            next_token_logits = logits[:, -1, :]

            if temperature == 0:
                next_token = torch.argmax(
                    next_token_logits,
                    dim=-1,
                    keepdim=True,
                )
            else:
                next_token_logits = next_token_logits / temperature

                if top_k is not None:
                    top_k = min(top_k, next_token_logits.size(-1))

                    values, _ = torch.topk(
                        next_token_logits,
                        k=top_k,
                        dim=-1,
                    )

                    min_values = values[:, -1].unsqueeze(-1)

                    next_token_logits = torch.where(
                        next_token_logits < min_values,
                        torch.full_like(next_token_logits, float("-inf")),
                        next_token_logits,
                    )

                if top_p is not None:
                    assert 0.0 < top_p <= 1.0

                    sorted_logits, sorted_indices = torch.sort(
                        next_token_logits,
                        descending=True,
                        dim=-1,
                    )

                    sorted_probs = torch.softmax(sorted_logits, dim=-1)
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                    sorted_indices_to_remove = cumulative_probs > top_p

                    sorted_indices_to_remove[:, 1:] = (
                        sorted_indices_to_remove[:, :-1].clone()
                    )
                    sorted_indices_to_remove[:, 0] = False

                    indices_to_remove = torch.zeros_like(
                        next_token_logits,
                        dtype=torch.bool,
                    )

                    indices_to_remove.scatter_(
                        dim=-1,
                        index=sorted_indices,
                        src=sorted_indices_to_remove,
                    )

                    next_token_logits = next_token_logits.masked_fill(
                        indices_to_remove,
                        float("-inf"),
                    )

                probs = torch.softmax(next_token_logits, dim=-1)

                next_token = torch.multinomial(
                    probs,
                    num_samples=1,
                )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1,
            )

            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break

        return input_ids

In [47]:
# -------------------------
# Create random variable-length input_ids and masks
# -------------------------

torch.manual_seed(42)

batch_size = 3
max_seq_len = 6
vocab_size = 100
pad_token_id = 0

lengths = torch.randint(
    low=1,
    high=max_seq_len + 1,
    size=(batch_size,)
)

input_ids = torch.randint(
    low=1,
    high=vocab_size,
    size=(batch_size, max_seq_len)
)

attention_mask = (
    torch.arange(max_seq_len).unsqueeze(0) < lengths.unsqueeze(1)
).long()

input_ids = input_ids.masked_fill(
    attention_mask == 0,
    pad_token_id,
)

labels = input_ids.clone()
labels = labels.masked_fill(attention_mask == 0, -100)

print("lengths:")
print(lengths)

print("\ninput_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)


# -------------------------
# Run MTP model
# -------------------------

config = Config(
    vocab_size=vocab_size,
    max_seq_len=max_seq_len,
    d_model=32,
    n_heads=4,
    n_layers=2,
    dropout=0.0,
    pad_token_id=pad_token_id,
    num_future_tokens=4,
    mtp_loss_weight=1.0,
)

model = MTPGPTDecoder(config)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

logits = outputs["logits"]
mtp_logits = outputs["mtp_logits"]
loss = outputs["loss"]
losses_by_depth = outputs["losses_by_depth"]

print("\nMain next-token logits shape:")
print(logits.shape)

print("\nAll MTP logits shapes:")
for i, depth_logits in enumerate(mtp_logits, start=1):
    print(f"depth {i} logits shape:", depth_logits.shape)

print("\nLoss:")
print(loss)

print("\nLosses by depth:")
for name, value in losses_by_depth.items():
    print(name, value)


# -------------------------
# Verify backward pass
# -------------------------

loss.backward()

print("\nGradient exists for MTP head 1?")
print(model.mtp_heads[0].weight.grad is not None)

print("\nGradient exists for MTP head 2?")
print(model.mtp_heads[1].weight.grad is not None)

print("\nEmbedding gradient exists?")
print(model.embedding.token_embeddings.weight.grad is not None)


# -------------------------
# Generation test
# -------------------------

prompt_len = int(attention_mask[0].sum().item())
prompt = input_ids[0:1, :prompt_len]

print("\nPrompt:")
print(prompt)

generated = model.generate(
    input_ids=prompt,
    max_new_tokens=5,
    temperature=1.0,
    top_k=10,
    top_p=0.9,
)

print("\nGenerated:")
print(generated)

print("\nGenerated shape:")
print(generated.shape)

lengths:
tensor([1, 6, 5])

input_ids:
tensor([[59,  0,  0,  0,  0,  0],
        [ 3, 68, 77, 81, 82, 91],
        [23, 93, 59, 40, 32,  0]])

attention_mask:
tensor([[1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0]])

labels:
tensor([[  59, -100, -100, -100, -100, -100],
        [   3,   68,   77,   81,   82,   91],
        [  23,   93,   59,   40,   32, -100]])

Main next-token logits shape:
torch.Size([3, 6, 100])

All MTP logits shapes:
depth 1 logits shape: torch.Size([3, 6, 100])
depth 2 logits shape: torch.Size([3, 6, 100])
depth 3 logits shape: torch.Size([3, 6, 100])
depth 4 logits shape: torch.Size([3, 6, 100])

Loss:
tensor(9.6475, grad_fn=<AddBackward0>)

Losses by depth:
main_next_token_loss_depth_1 tensor(4.7346)
mtp_aux_loss_depth_2 tensor(4.8077)
mtp_aux_loss_depth_3 tensor(4.8194)
mtp_aux_loss_depth_4 tensor(5.1118)

Gradient exists for MTP head 1?
True

Gradient exists for MTP head 2?
True

Embedding gradient exists?
True

Prompt:
tensor([[59

In [48]:
logits = torch.rand(3, 10, 100)
input_ids = torch.randint(0, 100, (3, 10))
print(logits)
print(input_ids)

tensor([[[9.3583e-01, 9.7014e-01, 7.4944e-02,  ..., 2.5831e-01,
          1.6394e-01, 2.5713e-01],
         [8.6214e-01, 7.2100e-01, 5.5107e-01,  ..., 8.1072e-01,
          5.3128e-01, 3.8369e-01],
         [6.8413e-01, 5.7567e-01, 6.1047e-02,  ..., 6.2650e-01,
          6.7142e-01, 2.4056e-01],
         ...,
         [8.6884e-02, 6.7147e-01, 7.2614e-01,  ..., 7.5520e-01,
          3.1455e-01, 1.8852e-01],
         [1.7071e-01, 3.0724e-01, 6.3759e-01,  ..., 2.0359e-02,
          3.7661e-01, 2.9586e-02],
         [7.9088e-01, 5.3481e-01, 9.8385e-01,  ..., 4.8611e-01,
          2.7879e-01, 2.9957e-01]],

        [[5.9239e-01, 8.2822e-01, 8.3439e-01,  ..., 3.7388e-01,
          7.1195e-02, 4.4310e-01],
         [2.5376e-01, 9.6244e-02, 5.1564e-01,  ..., 4.4062e-01,
          4.9730e-02, 6.7561e-01],
         [9.1377e-03, 6.3738e-01, 4.9557e-01,  ..., 2.2705e-02,
          1.0871e-01, 2.5081e-01],
         ...,
         [6.6409e-01, 9.8561e-01, 7.2385e-01,  ..., 5.9173e-01,
          6.608

In [49]:
shift_logits = logits[:, :-1, :].contiguous()
shift_logits

tensor([[[9.3583e-01, 9.7014e-01, 7.4944e-02,  ..., 2.5831e-01,
          1.6394e-01, 2.5713e-01],
         [8.6214e-01, 7.2100e-01, 5.5107e-01,  ..., 8.1072e-01,
          5.3128e-01, 3.8369e-01],
         [6.8413e-01, 5.7567e-01, 6.1047e-02,  ..., 6.2650e-01,
          6.7142e-01, 2.4056e-01],
         ...,
         [6.2097e-01, 5.1666e-01, 1.8581e-01,  ..., 5.1068e-01,
          2.8193e-01, 6.9751e-01],
         [8.6884e-02, 6.7147e-01, 7.2614e-01,  ..., 7.5520e-01,
          3.1455e-01, 1.8852e-01],
         [1.7071e-01, 3.0724e-01, 6.3759e-01,  ..., 2.0359e-02,
          3.7661e-01, 2.9586e-02]],

        [[5.9239e-01, 8.2822e-01, 8.3439e-01,  ..., 3.7388e-01,
          7.1195e-02, 4.4310e-01],
         [2.5376e-01, 9.6244e-02, 5.1564e-01,  ..., 4.4062e-01,
          4.9730e-02, 6.7561e-01],
         [9.1377e-03, 6.3738e-01, 4.9557e-01,  ..., 2.2705e-02,
          1.0871e-01, 2.5081e-01],
         ...,
         [5.9131e-01, 1.9490e-02, 3.3565e-01,  ..., 5.1660e-01,
          5.536

In [50]:
shift_logits.view(-1, 100)

tensor([[9.3583e-01, 9.7014e-01, 7.4944e-02,  ..., 2.5831e-01, 1.6394e-01,
         2.5713e-01],
        [8.6214e-01, 7.2100e-01, 5.5107e-01,  ..., 8.1072e-01, 5.3128e-01,
         3.8369e-01],
        [6.8413e-01, 5.7567e-01, 6.1047e-02,  ..., 6.2650e-01, 6.7142e-01,
         2.4056e-01],
        ...,
        [2.0982e-01, 4.1471e-01, 5.9479e-01,  ..., 7.7197e-01, 1.6797e-01,
         5.6208e-01],
        [9.4874e-01, 2.2374e-01, 6.9579e-01,  ..., 3.3066e-01, 1.7869e-01,
         7.9286e-04],
        [8.3926e-01, 3.4394e-01, 7.2695e-01,  ..., 3.7578e-01, 3.4920e-01,
         9.4329e-01]])

In [51]:
shift_logits.view(-1, 100).shape

torch.Size([27, 100])

In [52]:
input_ids = torch.randint(0, 100, (3, 10))
print(input_ids)
input_ids = input_ids[:, 1:].contiguous()
print(input_ids)

tensor([[26, 57,  4, 65, 90,  2, 79, 43, 84, 72],
        [24, 45, 79, 72, 71, 75,  7, 31, 83, 15],
        [29, 98, 55, 45, 96, 97, 29, 45, 51,  8]])
tensor([[57,  4, 65, 90,  2, 79, 43, 84, 72],
        [45, 79, 72, 71, 75,  7, 31, 83, 15],
        [98, 55, 45, 96, 97, 29, 45, 51,  8]])


In [53]:
input_ids.view(-1)

tensor([57,  4, 65, 90,  2, 79, 43, 84, 72, 45, 79, 72, 71, 75,  7, 31, 83, 15,
        98, 55, 45, 96, 97, 29, 45, 51,  8])

In [54]:
input_ids = torch.randint(0, 100, (3, 10))
print(input_ids)

tensor([[81, 28, 29, 59, 69, 45, 12, 52, 68,  2],
        [72, 89, 74, 72, 75, 88, 87, 14, 51, 62],
        [91, 72, 58, 93, 13, 65, 28, 57, 46, 73]])


In [55]:
input_ids = input_ids[:, 2:]

In [56]:
input_ids

tensor([[29, 59, 69, 45, 12, 52, 68,  2],
        [74, 72, 75, 88, 87, 14, 51, 62],
        [58, 93, 13, 65, 28, 57, 46, 73]])

In [ ]:
# DeepSeek-V3 Multi Token Prediction 

# main hidden h_i^0 predicts x_{i+1}

# MTP module 1:
# combine h_i^0 with embedding(x_{i+1})
#         ↓ projection + Transformer block
# h_i^1 predicts x_{i+2}

# MTP module 2:
# combine h_i^1 with embedding(x_{i+2})
#         ↓ projection + Transformer block
# h_i^2 predicts x_{i+3}

In [57]:
@dataclass
class Config:
    vocab_size: int = 100
    max_seq_len: int = 6
    d_model: int = 32
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    pad_token_id: int = 0

    # DeepSeek-V3-style MTP
    mtp_num_layers: int = 2
    mtp_loss_weight: float = 0.1

    @property
    def d_head(self):
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads

In [ ]:
class DeepSeekMTPModule(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.proj = nn.Linear(
            2 * config.d_model,
            config.d_model,
            bias=False,
        )

        self.block = DecoderBlock(config)

    def forward(self, prev_hidden, future_token_emb, attention_mask=None):
        """
        prev_hidden:
            [batch_size, seq_len, d_model]

        future_token_emb:
            [batch_size, seq_len, d_model]

        attention_mask:
            [batch_size, seq_len]

        returns:
            hidden:
                [batch_size, seq_len, d_model]
        """

        x = torch.cat(
            [prev_hidden, future_token_emb],
            dim=-1,
        )
        # [B, T, 2 * D]

        x = self.proj(x)
        # [B, T, D]

        x = self.block(
            x,
            attention_mask=attention_mask,
        )
        # [B, T, D]

        return x

In [ ]:
class DeepSeekMTPGPTDecoder(nn.Module):
    def __init__(self, config: Config):
        super().__init__()

        self.config = config

        self.embedding = TokenEmbeddingLayer(config)

        self.blocks = nn.ModuleList([
            DecoderBlock(config)
            for _ in range(config.n_layers)
        ])

        self.final_ln = nn.LayerNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False,
        )

        # Main model weight tying
        self.lm_head.weight = self.embedding.token_embeddings.weight

        self.mtp_modules = nn.ModuleList([
            DeepSeekMTPModule(config)
            for _ in range(config.mtp_num_layers)
        ])

    def forward(self, input_ids, attention_mask=None, labels=None):
        """
        input_ids:
            [B, T]

        attention_mask:
            [B, T]

        labels:
            [B, T], integer token IDs with -100 for ignored positions

        returns:
            logits:
                main next-token logits, [B, T, vocab_size]

            loss:
                main next-token loss + weighted MTP auxiliary loss
        """

        batch_size, seq_len = input_ids.shape
        assert seq_len <= self.config.max_seq_len

        # -------------------------
        # Main decoder trunk
        # -------------------------

        x = self.embedding(input_ids)

        # If your older embedding layer returns (x, position_ids),
        # keep this line. If it returns only x, this does nothing.
        if isinstance(x, tuple):
            x = x[0]

        for block in self.blocks:
            x = block(
                x,
                attention_mask=attention_mask,
            )

        main_hidden = self.final_ln(x)
        # [B, T, D]

        logits = self.lm_head(main_hidden)
        # [B, T, vocab_size]

        loss = None
        main_loss = None
        mtp_loss = None
        losses_by_depth = {}

        if labels is not None:
            # -------------------------
            # Main next-token loss
            # logits at position i predict token i+1
            # -------------------------

            main_shift_logits = logits[:, :-1, :]
            main_shift_labels = labels[:, 1:]

            main_loss = F.cross_entropy(
                main_shift_logits.reshape(-1, self.config.vocab_size),
                main_shift_labels.reshape(-1),
                ignore_index=-100,
            )

            losses_by_depth["main_next_token_loss"] = main_loss.detach()

            # -------------------------
            # DeepSeek-style sequential MTP losses
            # -------------------------

            mtp_losses = []

            prev_hidden = main_hidden
            # depth 0 representation: h^0

            if attention_mask is None:
                prev_valid_mask = torch.ones(
                    batch_size,
                    seq_len,
                    device=input_ids.device,
                    dtype=torch.long,
                )
            else:
                prev_valid_mask = attention_mask.long()

            for depth, mtp_module in enumerate(self.mtp_modules, start=1):
                # depth = 1:
                #   combine h_i^0 with embedding(x_{i+1})
                #   predict x_{i+2}
                #
                # depth = 2:
                #   combine h_i^1 with embedding(x_{i+2})
                #   predict x_{i+3}

                if seq_len <= depth:
                    break

                # At depth 1, we combine: h_i^0 with embedding(x_{i+1}) 
                    # h0^0 + emb(x1) → h0^1 → predict x2
                    # h1^0 + emb(x2) → h1^1 → predict x3
                    # h2^0 + emb(x3) → h2^1 → predict x4
                    # h3^0 + emb(x4) → h3^1 → predict x5
                    # h4^0 + emb(x5) → h4^1 → no target, because x6 does not exist and thats the reason we slice prev_hidden[:, :-1, :] where prev_hidden is h0 

                #  # At depth 2,
                    # h0^1 + emb(x2) → h0^2 → predict x3
                    # h1^1 + emb(x3) → h1^2 → predict x4
                    # h2^1 + emb(x4) → h2^2 → predict x5
                    # h3^1 + emb(x5) → h3^2 → no target, because x6 does not exist and that reason we slice [:, :-1, :]  where prev_hidden is h1
                
                prev_hidden_for_depth = prev_hidden[:, :-1, :]
                # [B, T-depth, D]

                future_token_ids = input_ids[:, depth:]
                # [B, T-depth]

                future_token_emb = self.embedding.token_embeddings(
                    future_token_ids
                )
                # [B, T-depth, D]

                if attention_mask is not None:
                    current_valid_mask = (
                        prev_valid_mask[:, :-1].bool()
                        & attention_mask[:, depth:].bool()
                    ).long()
                else:
                    current_valid_mask = None

                depth_hidden = mtp_module(
                    prev_hidden=prev_hidden_for_depth,
                    future_token_emb=future_token_emb,
                    attention_mask=current_valid_mask,
                )
                # [B, T-depth, D]

                depth_logits = self.lm_head(depth_hidden)
                # [B, T-depth, vocab_size]

                # MTP module at depth d predicts token i+d+1.
                # So for depth=1, predict labels[:, 2:].
                # For depth=2, predict labels[:, 3:].
                if seq_len > depth + 1:
                    mtp_shift_logits = depth_logits[:, :-1, :].contiguous()
                    mtp_shift_labels = labels[:, depth + 1:].contiguous()

                    if (mtp_shift_labels != -100).any():
                        loss_depth = F.cross_entropy(
                            mtp_shift_logits.reshape(
                                -1,
                                self.config.vocab_size,
                            ),
                            mtp_shift_labels.reshape(-1),
                            ignore_index=-100,
                        )

                        mtp_losses.append(loss_depth)
                        losses_by_depth[f"mtp_loss_depth_{depth}"] = (
                            loss_depth.detach()
                        )

                prev_hidden = depth_hidden

                if current_valid_mask is not None:
                    prev_valid_mask = current_valid_mask

            if len(mtp_losses) > 0:
                mtp_loss = torch.stack(mtp_losses).mean()
                loss = main_loss + self.config.mtp_loss_weight * mtp_loss
            else:
                mtp_loss = torch.tensor(
                    0.0,
                    device=input_ids.device,
                    dtype=main_loss.dtype,
                )
                loss = main_loss

        return {
            "logits": logits,
            "loss": loss,
            "main_loss": main_loss,
            "mtp_loss": mtp_loss,
            "losses_by_depth": losses_by_depth,
        }

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
        eos_token_id: int | None = None,
    ):
        """
        Normal autoregressive generation.

        DeepSeek-V3 says MTP modules can be discarded during normal inference.
        So this uses only the main model logits.
        """

        self.eval()

        for _ in range(max_new_tokens):
            input_ids_cond = input_ids[:, -self.config.max_seq_len:]

            attention_mask = (
                input_ids_cond != self.config.pad_token_id
            ).long()

            outputs = self(
                input_ids=input_ids_cond,
                attention_mask=attention_mask,
                labels=None,
            )

            next_token_logits = outputs["logits"][:, -1, :]

            if temperature == 0:
                next_token = torch.argmax(
                    next_token_logits,
                    dim=-1,
                    keepdim=True,
                )
            else:
                next_token_logits = next_token_logits / temperature

                if top_k is not None:
                    top_k = min(top_k, next_token_logits.size(-1))

                    values, _ = torch.topk(
                        next_token_logits,
                        k=top_k,
                        dim=-1,
                    )

                    min_values = values[:, -1].unsqueeze(-1)

                    next_token_logits = torch.where(
                        next_token_logits < min_values,
                        torch.full_like(next_token_logits, float("-inf")),
                        next_token_logits,
                    )

                if top_p is not None:
                    assert 0.0 < top_p <= 1.0

                    sorted_logits, sorted_indices = torch.sort(
                        next_token_logits,
                        descending=True,
                        dim=-1,
                    )

                    sorted_probs = torch.softmax(sorted_logits, dim=-1)
                    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                    sorted_indices_to_remove = cumulative_probs > top_p

                    sorted_indices_to_remove[:, 1:] = (
                        sorted_indices_to_remove[:, :-1].clone()
                    )
                    sorted_indices_to_remove[:, 0] = False

                    indices_to_remove = torch.zeros_like(
                        next_token_logits,
                        dtype=torch.bool,
                    )

                    indices_to_remove.scatter_(
                        dim=-1,
                        index=sorted_indices,
                        src=sorted_indices_to_remove,
                    )

                    next_token_logits = next_token_logits.masked_fill(
                        indices_to_remove,
                        float("-inf"),
                    )

                probs = torch.softmax(next_token_logits, dim=-1)

                next_token = torch.multinomial(
                    probs,
                    num_samples=1,
                )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1,
            )

            if eos_token_id is not None:
                if torch.all(next_token.squeeze(-1) == eos_token_id):
                    break

        return input_ids